# Irodori-TTS 音声合成 Colab ノートブック

[Irodori-TTS](https://github.com/Aratako/Irodori-TTS) を使用したローカル日本語音声合成ノートブックです。  
**テキストプロンプトで声質・感情・話し方を自在に指定**できます（参照音声不要）。

## 前提条件
- ランタイムを **T4 GPU** に設定してください（ランタイム → ランタイムのタイプを変更）
- 初回実行時にモデル（約 2GB）を HuggingFace からダウンロードします

## モデル
| チェックポイント | 特徴 |
|---|---|
| `Aratako/Irodori-TTS-500M-v2-VoiceDesign` | テキストキャプションで声質・感情を制御（参照音声不要） |

In [ ]:
# ── セットアップ（初回のみ数分かかります）──────────────────────────────
import subprocess, sys, os

# GPU 確認
res = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if res.returncode == 0:
    print(f"✅ GPU 検出: {res.stdout.strip()}")
else:
    raise RuntimeError("GPU が見つかりません。ランタイムのタイプを T4 GPU に変更してください。")

# Irodori-TTS リポジトリ取得
REPO_DIR = '/content/Irodori-TTS'
if not os.path.exists(REPO_DIR):
    print("\n📥 Irodori-TTS をクローン中...")
    subprocess.run(
        ['git', 'clone', 'https://github.com/Aratako/Irodori-TTS.git', REPO_DIR],
        check=True
    )
    print("✅ クローン完了")
else:
    print(f"\n✅ {REPO_DIR} は既に存在します")

# 依存関係インストール
# （Colab に既存の PyTorch はそのまま利用。git 依存の dacvae / silentcipher を含む）
print("\n📦 依存関係をインストール中（初回は数分かかります）...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
    check=True
)
print("✅ インストール完了")

# Python パスにリポジトリを追加
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("\n🎉 セットアップ完了")

In [ ]:
# ── モデル読み込み（セッション中に 1 度だけ実行）──────────────────────
from huggingface_hub import hf_hub_download
from irodori_tts.inference_runtime import RuntimeKey, SamplingRequest, get_cached_runtime

HF_REPO         = "Aratako/Irodori-TTS-500M-v2-VoiceDesign"
MODEL_DEVICE    = "cuda"
CODEC_REPO      = "Aratako/Semantic-DACVAE-Japanese-32dim"
MODEL_PRECISION = "bf16"
CODEC_PRECISION = "bf16"

# RuntimeKey.checkpoint はローカルパスのみ受け付けるため、先に HF からダウンロードする
print(f"📥 モデルをダウンロード中: {HF_REPO}")
print("   初回は HuggingFace からダウンロードします（約 2GB）...\n")
local_checkpoint = hf_hub_download(repo_id=HF_REPO, filename="model.safetensors")
print(f"✅ ダウンロード完了: {local_checkpoint}\n")

print("🔄 モデルを読み込み中...")
runtime_key = RuntimeKey(
    checkpoint=local_checkpoint,
    model_device=MODEL_DEVICE,
    codec_repo=CODEC_REPO,
    model_precision=MODEL_PRECISION,
    codec_device=MODEL_DEVICE,
    codec_precision=CODEC_PRECISION,
    compile_model=False,
    compile_dynamic=False,
)

runtime, _ = get_cached_runtime(runtime_key)
print("\n✅ モデル読み込み完了")

In [ ]:
# ── 音声合成（基本）────────────────────────────────────────────────────
import datetime, os
import torchaudio
from IPython.display import Audio, display

LOCAL_AUDIO_DIR = "/content/audio_output"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

# ── 読み上げテキスト ────────────────────────────────────────────────
TEXT = "こんにちは。今日はどんなお話をしましょうか？もしよければ、ゆっくりお話ししますね。"

# ── 声質・感情プロンプト（VoiceDesign キャプション）──────────────────
# 日本語・英語・絵文字で声の雰囲気、感情、話し方を自由に記述します
CAPTION = "落ち着いた女性の声で、近い距離感でやわらかく自然に読み上げてください。"

# ── 合成パラメータ ──────────────────────────────────────────────────
NUM_STEPS         = 40    # 拡散ステップ数（多いほど高品質・低速。20〜60 が実用域）
CFG_SCALE_TEXT    = 2.0   # テキスト誘導強度
CFG_SCALE_CAPTION = 2.0   # キャプション誘導強度
SEED              = 42

# ── 合成実行 ────────────────────────────────────────────────────────
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{LOCAL_AUDIO_DIR}/irodori_{ts}.wav"

print(f"🎙️ 合成中...")
print(f"   テキスト  : {TEXT}")
print(f"   キャプション: {CAPTION}\n")

request = SamplingRequest(
    text=TEXT,
    caption=CAPTION,
    no_ref=True,
    num_steps=NUM_STEPS,
    seed=SEED,
    cfg_scale_text=CFG_SCALE_TEXT,
    cfg_scale_caption=CFG_SCALE_CAPTION,
)

result = runtime.synthesize(request)

# 保存（torchaudio で WAV 出力）
audio = result.audio.cpu().float().unsqueeze(0)  # (1, samples)
torchaudio.save(output_path, audio, result.sample_rate)

print(f"✅ 保存: {output_path}")
display(Audio(output_path, autoplay=False))

In [ ]:
# ── VoiceDesign スタイル比較（複数のキャプションで同じテキストを読み上げ）──
COMPARE_TEXT = "今日は少し疲れましたが、お話できて嬉しいです。"

CAPTION_STYLES = [
    ("明るい女性",
     "明るく元気な若い女性の声で、笑顔が伝わるように読み上げてください。😊"),
    ("落ち着いた男性",
     "落ち着いた中年男性の声で、ゆっくりと丁寧に読み上げてください。"),
    ("疲れた声",
     "少し疲れた様子で、感情を込めてつぶやくように読み上げてください。😔"),
    ("アナウンサー",
     "NHKのアナウンサーのように、明瞭で標準的な発音で読み上げてください。"),
]

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

for style_name, caption in CAPTION_STYLES:
    print(f"\n🎙️ スタイル: {style_name}")
    print(f"   {caption}")

    output_path = f"{LOCAL_AUDIO_DIR}/style_{style_name}_{ts}.wav"

    request = SamplingRequest(
        text=COMPARE_TEXT,
        caption=caption,
        no_ref=True,
        num_steps=NUM_STEPS,
        seed=SEED,
        cfg_scale_text=CFG_SCALE_TEXT,
        cfg_scale_caption=CFG_SCALE_CAPTION,
    )

    result = runtime.synthesize(request)

    audio = result.audio.cpu().float().unsqueeze(0)
    torchaudio.save(output_path, audio, result.sample_rate)

    print(f"   ✅ 保存: {output_path}")
    display(Audio(output_path, autoplay=False))